# Transfer Learning with TensorFlow/Keras (Student-Run Tutorial)

This notebook is designed to be **run end-to-end by students**, without instructor intervention.

## Learning objectives
By the end of this notebook, you should be able to:

1. Explain what **transfer learning** is and why it works.
2. Implement **feature extraction** (frozen backbone + new classifier head).
3. Implement **fine-tuning** (unfreeze part/all of the backbone and train with a small learning rate).
4. Diagnose common pitfalls (wrong preprocessing, too-large learning rate, mismatch between backbone and input scale).
5. Apply the same pipeline to a related task (**CIFAR-100**) as a copy/paste challenge.

---

## Narrative connection to representation learning
A pretrained CNN (trained on ImageNet) can be seen as a **representation learner**:
- early layers learn generic visual features (edges/textures),
- later layers learn higher-level features (parts/shapes),
- the final layer maps features to labels.

Transfer learning reuses these learned representations for a new task.


## 0. Setup

Run the next cell to import dependencies and check whether a GPU is available.

> If you have no GPU, everything will still work, but it may run more slowly. If it is too slow, use the *MobileNetV2* extension later.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)


## 1. Load CIFAR-10

CIFAR-10 is a standard small image dataset with 10 classes. Images are **32×32** RGB.

We will transfer a model pretrained on ImageNet (which typically expects ~224×224 images), so we will later **resize** images and apply the backbone-specific preprocessing.


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Train:", x_train.shape, y_train.shape)
print("Test: ", x_test.shape, y_test.shape)

class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]


### (Optional) Visualize a few samples

This is just to make the dataset concrete.


In [ ]:
plt.figure(figsize=(8,4))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[int(y_train[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()


## 2. Build a `tf.data` pipeline + preprocessing

**Key idea:** pretrained models expect a specific input preprocessing function.

We will:
1. Resize images to `IMG_SIZE×IMG_SIZE`.
2. Apply `tf.keras.applications.resnet.preprocess_input`.

> If you switch backbones later (e.g., MobileNetV2), you must also switch the preprocessing function accordingly.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64

def preprocess_for_resnet(x, y):
    # Resize: CIFAR-10 is 32x32, ImageNet backbones typically expect ~224x224.
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    # IMPORTANT: backbone-specific preprocessing
    x = tf.keras.applications.resnet.preprocess_input(x)
    return x, y

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(10000).batch(BATCH_SIZE).map(preprocess_for_resnet).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.batch(BATCH_SIZE).map(preprocess_for_resnet).prefetch(tf.data.AUTOTUNE)

# Sanity check: one batch
for xb, yb in train_ds.take(1):
    print("Batch X:", xb.shape, "Batch y:", yb.shape)


## 3. Build the transfer-learning model

We will use a pretrained ImageNet backbone:
- `include_top=False` means we **remove** the original 1000-class ImageNet classifier head.
- We add a new head for CIFAR-10 (10 classes).

We start with **feature extraction** mode: the backbone is **frozen** (not trainable).


In [ ]:
base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Feature extraction: freeze the backbone
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)  # turns feature maps into a vector
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()


## 4. Train the classifier head (feature extraction)

Only the new head trains; the pretrained backbone stays fixed.

**Question (write your answer below):**
- Why might freezing the backbone be a good idea when you have limited data?


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fe = model.fit(train_ds, epochs=5, validation_data=test_ds)


In [ ]:
plt.figure()
plt.plot(history_fe.history["accuracy"], label="train acc")
plt.plot(history_fe.history["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.show()


## 5. Fine-tuning

Now we allow the backbone to adapt to CIFAR-10.

Important practical rules:
- Use a **smaller learning rate** during fine-tuning.
- Optionally fine-tune only the *later* layers (often safer).

Below we unfreeze the full backbone for simplicity. If training becomes unstable or slow, fine-tune only the last blocks (see the commented code).

**Question (write your answer below):**
- Why do we reduce the learning rate for fine-tuning?


In [ ]:
# Unfreeze the backbone
base_model.trainable = True

# Optional: freeze early layers, fine-tune only later layers
# for layer in base_model.layers[:140]:
#     layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(train_ds, epochs=3, validation_data=test_ds)


In [ ]:
plt.figure()
plt.plot(history_ft.history["accuracy"], label="train acc")
plt.plot(history_ft.history["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.show()


## 6. Final evaluation

We evaluate on the test set.

**Interpretation prompt:**
- Compare your feature-extraction validation accuracy vs your fine-tuning accuracy.
- Did fine-tuning help? If yes, why might that be the case?


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.3f}")


## 7. Common pitfalls checklist (read this if results look wrong)

If your accuracy is unexpectedly poor:

1. **Preprocessing mismatch**  
   Ensure you used the correct `preprocess_input` function for the chosen backbone.

2. **Learning rate too large during fine-tuning**  
   Try `1e-5` or even `1e-6`.

3. **Training too slow (CPU-only)**  
   Use the MobileNetV2 extension below, or reduce `IMG_SIZE` (e.g., 160).

4. **Overfitting**  
   Increase dropout, use data augmentation, or fine-tune fewer layers.


## 8. Optional extension: switch to a faster backbone (MobileNetV2)

If you are CPU-limited, MobileNetV2 is usually much faster.

Instructions:
- Replace the ResNet50 backbone with MobileNetV2.
- Update preprocessing to `tf.keras.applications.mobilenet_v2.preprocess_input`.

The cells below are optional and do not affect the main tutorial.


In [ ]:
# OPTIONAL: MobileNetV2 version (faster)
# Uncomment and run this cell to rebuild the model with MobileNetV2.

# base_model = tf.keras.applications.MobileNetV2(
#     weights="imagenet",
#     include_top=False,
#     input_shape=(IMG_SIZE, IMG_SIZE, 3)
# )
# base_model.trainable = False

# def preprocess_for_mobilenet(x, y):
#     x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
#     x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
#     return x, y

# train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
# train_ds = train_ds.shuffle(10000).batch(BATCH_SIZE).map(preprocess_for_mobilenet).prefetch(tf.data.AUTOTUNE)

# test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
# test_ds = test_ds.batch(BATCH_SIZE).map(preprocess_for_mobilenet).prefetch(tf.data.AUTOTUNE)

# inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
# x = base_model(inputs, training=False)
# x = tf.keras.layers.GlobalAveragePooling2D()(x)
# x = tf.keras.layers.Dropout(0.2)(x)
# outputs = tf.keras.layers.Dense(10, activation="softmax")(x)
# model = tf.keras.Model(inputs, outputs)
# model.summary()


# Student Challenge (copy/paste + small edits)

## Challenge A: Transfer learning on CIFAR-100

Goal: Repeat the same pipeline using CIFAR-100 (100 classes).

Tasks:
1. Load CIFAR-100 instead of CIFAR-10.
2. Keep the same preprocessing and data pipeline pattern.
3. Replace the classifier head: `Dense(100, softmax)`.
4. Train in two modes:
   - feature extraction (frozen backbone)
   - fine-tuning (unfrozen backbone, small LR)
5. Compare results and answer:
   - Which approach works better?
   - Why might fine-tuning help more (or less) on CIFAR-100?

### Hints
- `tf.keras.datasets.cifar100.load_data()`
- change the head to `Dense(100, activation="softmax")`

---

## Challenge B (optional): Partial fine-tuning

Instead of unfreezing the whole backbone, fine-tune only later layers.

Example idea:
- freeze layers `[:140]` and fine-tune `layers[140:]`.

Write down what you changed and what happened to validation accuracy/training time.


In [ ]:
# --- Challenge A starter code (students fill in) ---

# 1) Load CIFAR-100
# (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# 2) Rebuild train_ds / test_ds (same pattern as before)
#    Make sure your preprocess function matches your backbone (ResNet50 -> resnet.preprocess_input)

# 3) Rebuild model with Dense(100)
# outputs = tf.keras.layers.Dense(100, activation="softmax")(x)

# 4) Train: feature extraction
# base_model.trainable = False
# model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
#               loss="sparse_categorical_crossentropy",
#               metrics=["accuracy"])
# model.fit(train_ds, epochs=5, validation_data=test_ds)

# 5) Train: fine-tuning
# base_model.trainable = True
# model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
#               loss="sparse_categorical_crossentropy",
#               metrics=["accuracy"])
# model.fit(train_ds, epochs=3, validation_data=test_ds)

# 6) Evaluate + compare
# model.evaluate(test_ds)
